In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from few.amplitude.ampinterp2d import  AmpInterpKerrEccEq_nex, AmpInterp2D
from few.utils.baseclasses import KerrEccentricEquatorial_nex, BackendLike
from few.amplitude.base import AmplitudeBase
import os
import pathlib
from copy import deepcopy
from typing import List, Optional, Union
import h5py
from few import get_file_manager

In [34]:
class AmpInterpKerrEccEq_nex(AmplitudeBase, KerrEccentricEquatorial_nex):
    """Calculate Teukolsky amplitudes in the Kerr eccentric equatorial regime with a bicubic spline + linear
    interpolation scheme.

    When called with arguments :math:`(a, p, e, xI)`, these parameters are transformed into a set of
    interpolation coordinates and the bicubic spline interpolant is evaluated at these coordinates for
    all sets of coefficients. To interpolate in the :math"`a` direction, the bicubic spline is evaluated at
    the adjacent grid points and a linear interpolation is performed.

    This module is available for GPU and CPU.

    args:
        fp: The coefficients file name in `file_directory`.
        **kwargs: Optional keyword arguments for the base classes:
            :class:`few.utils.baseclasses.AmplitudeBase`,
            :class:`few.utils.baseclasses.KerrEccentricEquatorialv2`.
    """

    filename: str

    spin_information_holder_A: list[AmpInterp2D]

    z_values: np.ndarray

    def __init__(
        self,
        filename: Optional[str] = None,
        downsample_Z=1,
        force_backend: BackendLike = None,
        **kwargs,
    ):
        AmplitudeBase.__init__(self)

        self.filename = (
            "ZNAmps_l10_m10_n92__extremal.h5" if filename is None else filename
        )

        from few import get_file_manager

        file_path = get_file_manager().get_file(self.filename)

        with h5py.File(file_path, "r") as f:
            mode_indices = f["modes"]["mode_indices"][()]

        KerrEccentricEquatorial_nex.__init__(self, mode_indices=mode_indices, force_backend=force_backend, **kwargs)

        with h5py.File(file_path, "r") as f:
            regionA = f["regionA"]
            coeffsA = regionA["CoeffsRegionA"][()]
            w_knots = regionA["w_knots"][()]
            u_knots = regionA["u_knots"][()]
            z_knots = regionA["z_knots"][()]
            # paramsA = regionA["ParamsRegionA"][()]

            z_knots = z_knots[::downsample_Z]
            coeffsA = coeffsA[::downsample_Z]

            self.spin_information_holder_A = [
                self.build_with_same_backend(
                    AmpInterp2D,
                    args=[
                        w_knots,
                        u_knots,
                        coeffsA[i],
                        self.l_arr,
                        self.m_arr,
                        self.mode_arr[:, 2],
                        self.mode_arr[:, 3],
                    ],
                )
                for i in range(z_knots.size)
            ]

            try:
                regionB = f["regionB"]
                coeffsB = regionB["CoeffsRegionB"][()]

                w_knots = regionB["w_knots"][()]
                u_knots = regionB["u_knots"][()]
                z_knots = regionB["z_knots"][()]

                z_knots = z_knots[::downsample_Z]

                coeffsB = coeffsB[::downsample_Z]

                self.spin_information_holder_B = [
                    self.build_with_same_backend(
                        AmpInterp2D,
                        args=[
                            w_knots,
                            u_knots,
                            coeffsB[i],
                            self.l_arr,
                            self.m_arr,
                            self.mode_arr[:, 2],
                            self.mode_arr[:, 3],
                        ],
                    )
                    for i in range(z_knots.size)
                ]
            except KeyError:
                pass

        self.z_values = z_knots

    def _evaluate_interpolant_at_index(self, index, region_A_mask, w, u, mode_indexes):
        z_out = self.xp.zeros(
            (region_A_mask.size, self.num_modes_eval), dtype=self.xp.complex128
        )

        if self.xp.any(region_A_mask):
            z_out[region_A_mask, :] = self.spin_information_holder_A[index](
                w[region_A_mask], u[region_A_mask], mode_indexes=mode_indexes
            )

        if self.xp.any(~region_A_mask):
            z_out[~region_A_mask, :] = self.spin_information_holder_B[index](
                w[~region_A_mask], u[~region_A_mask], mode_indexes=mode_indexes
            )

        return z_out

    def get_amplitudes(
        self,
        a: float,
        p: Union[float, np.ndarray],
        e: Union[float, np.ndarray],
        xI: Union[float, np.ndarray],
        specific_modes: Optional[np.ndarray] = None,
    ) -> np.ndarray:
        """
        Generate Teukolsky amplitudes for a given set of parameters.

        Args:
            a: Dimensionless spin parameter of MBH.
            p: Dimensionless semi-latus rectum.
            e: Eccentricity.
            xI: Cosine of orbital inclination. Only :math:`|x_I| = 1` is currently supported.
            specific_modes: Indices of modes to be generated (optional; defaults to all modes).
        Returns:
            An array of complex mode amplitudes.
        """
        if specific_modes is None:
            specific_modes = self.xp.arange(self.num_teuk_modes)

        assert self.xp.all(a == a[0]), "All spins must be the same value."
        assert self.xp.all(a * xI <= 0.0) or self.xp.all(
            a * xI >= 0.0
        )  # either all prograde or all retrograde
        assert self.xp.all(self.xp.abs(xI) == 1.0)  # all equatorial

        xI_in = self.xp.ones_like(p) * xI

        signed_spin = a * xI_in
        a_in = self.xp.ones(p.size) * signed_spin
        xI_in = self.xp.abs(xI_in)

        try:
            u, w, y, z, region_mask = kerrecceq_forward_map_nex(
                a_in.get(),
                p.get(),
                e.get(),
                xI_in.get(),
                return_mask=True,
                kind="amplitude",
            )

        except AttributeError:
            u, w, y, z, region_mask = kerrecceq_forward_map_nex(
                a_in,
                p,
                e,
                xI_in,
                return_mask=True,
                kind="amplitude",
            )
        z_check = z[0].item()

        region_mask = self.xp.asarray(region_mask)
        u = self.xp.asarray(u)
        w = self.xp.asarray(w)
        z = self.xp.asarray(z)
        self.z_values = self.xp.asarray(self.z_values)

        for elem in [u, w, z]:
            if self.xp.any((elem < 0) | (elem > 1)):
                raise ValueError("Amplitude interpolant accessed out-of-bounds.")

        if z_check in self.z_values:
            try:
                ind_1 = self.xp.where(self.z_values == z_check)[0].get()[0]
            except AttributeError:
                ind_1 = self.xp.where(self.z_values == z_check)[0][0]

            Amp_z = self._evaluate_interpolant_at_index(
                ind_1, region_mask, w, u, mode_indexes=specific_modes
            )

        else:
            try:
                ind_above = self.xp.where(self.z_values > z_check)[0].get()[0]
            except AttributeError:
                ind_above = self.xp.where(self.z_values > z_check)[0][0]
            ind_below = ind_above - 1
            assert ind_above < len(self.z_values)
            assert ind_below >= 0

            z_above = self.z_values[ind_above]
            Amp_above = self._evaluate_interpolant_at_index(
                ind_above, region_mask, w, u, specific_modes
            )

            z_below = self.z_values[ind_below]
            Amp_below = self._evaluate_interpolant_at_index(
                ind_below, region_mask, w, u, specific_modes
            )

            Amp_z = ((Amp_above - Amp_below) / (z_above - z_below)) * (
                z_check - z_below
            ) + Amp_below

        return Amp_z

In [35]:
AmpInterpKerrEccEq_nex().get_amplitudes(0.99,6,0.2,1)

IndexError: index -247 is out of bounds for axis 3 with size 185

In [37]:
filename = (
    "ZNAmps_l10_m10_n92__extremal.h5" 
)


file_path = get_file_manager().get_file(filename)

with h5py.File(file_path, "r") as f:
    mode_indices = f["modes"]["mode_indices"][()]
    for i in mode_indices:
        print(i)



with h5py.File(file_path, "r") as f:
    regionA = f["regionA"]
    coeffsA = regionA["CoeffsRegionA"][()]
    w_knots = regionA["w_knots"][()]
    u_knots = regionA["u_knots"][()]
    z_knots = regionA["z_knots"][()]
    # paramsA = regionA["ParamsRegionA"][()]

    z_knots = z_knots
    coeffsA = coeffsA


[  2   0   0 -92]
[  2   0   0 -91]
[  2   0   0 -90]
[  2   0   0 -89]
[  2   0   0 -88]
[  2   0   0 -87]
[  2   0   0 -86]
[  2   0   0 -85]
[  2   0   0 -84]
[  2   0   0 -83]
[  2   0   0 -82]
[  2   0   0 -81]
[  2   0   0 -80]
[  2   0   0 -79]
[  2   0   0 -78]
[  2   0   0 -77]
[  2   0   0 -76]
[  2   0   0 -75]
[  2   0   0 -74]
[  2   0   0 -73]
[  2   0   0 -72]
[  2   0   0 -71]
[  2   0   0 -70]
[  2   0   0 -69]
[  2   0   0 -68]
[  2   0   0 -67]
[  2   0   0 -66]
[  2   0   0 -65]
[  2   0   0 -64]
[  2   0   0 -63]
[  2   0   0 -62]
[  2   0   0 -61]
[  2   0   0 -60]
[  2   0   0 -59]
[  2   0   0 -58]
[  2   0   0 -57]
[  2   0   0 -56]
[  2   0   0 -55]
[  2   0   0 -54]
[  2   0   0 -53]
[  2   0   0 -52]
[  2   0   0 -51]
[  2   0   0 -50]
[  2   0   0 -49]
[  2   0   0 -48]
[  2   0   0 -47]
[  2   0   0 -46]
[  2   0   0 -45]
[  2   0   0 -44]
[  2   0   0 -43]
[  2   0   0 -42]
[  2   0   0 -41]
[  2   0   0 -40]
[  2   0   0 -39]
[  2   0   0 -38]
[  2   0  

In [32]:
coeffsA

array([[[[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
           0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
         [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
           0.00000000e+00,  0.00000000e+00,  0.00000000e+00]],

        [[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
           0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
         [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
           0.00000000e+00,  0.00000000e+00,  0.00000000e+00]],

        [[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
           0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
         [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
           0.00000000e+00,  0.00000000e+00,  0.00000000e+00]],

        ...,

        [[ 4.03150719e-44, -2.78333961e-41, -2.73965521e-41, ...,
           3.43448936e-09, -8.31997619e-09, -1.76608945e-09],
         [ 2.56141859e-44, -1.28152322e-40,  1.31254000e-41, ...,
          -1.13190

In [38]:
np.max(mode_indices, axis=0)

array([10, 10,  0, 92])